In [1]:
from dataclasses import dataclass
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

from gnsskit import RtkConfig, read_pos, run_rnx2rtkp, utc_to_gpst
from gnsskit.baseline import baselines_to_df, process_baseline
from gnsskit.pos import read_ref_pos


@dataclass
class RinexPaths:
    obs_file: Pathz
    nav_files: tuple[Path, ...]


def get_rinex_paths(
    rinex_dir: Path, point: str, nav_suff: list[str] | None = None
) -> RinexPaths:
    """Get the paths to the RINEX observation and navigation files for a given point."""

    if nav_suff is None:
        nav_suff = ["n", "g", "l", "f"]

    obs_file = list(rinex_dir.glob(f"{point}.*o"))

    if not obs_file:
        raise FileNotFoundError(f"No RINEX observation file found for point {point}.")

    obs_file = obs_file[0]

    nav_files = []
    for suff in nav_suff:
        nav_files.extend(list(rinex_dir.glob(f"{point}.*{suff}")))

    if not nav_files:
        raise FileNotFoundError(f"No RINEX navigation file found for point {point}.")

    return RinexPaths(obs_file=obs_file, nav_files=tuple(nav_files))

## Punti di stazione:

1. 063604: punto noto
2. 1002
3. 1005
4. 1006

In [ ]:
# Paths
rinex_dir = Path("./rinex")
PCV_file = Path("./aux/igs20.atx")
sp3_path = Path("./aux/COD0MGXFIN_20262800000_01D_05M_ORB.SP3")
clk_path = Path("./aux/COD0MGXFIN_20262800000_01D_30S_CLK.CLK")

baseline_dir = Path("./output")
baseline_dir.mkdir(exist_ok=True)

files = {
    "1002": get_rinex_paths(rinex_dir, "10022800", nav_suff=["n", "g", "l"]),
    "1005": get_rinex_paths(rinex_dir, "10052800", nav_suff=["n", "g", "l"]),
    "1006": get_rinex_paths(rinex_dir, "10062800", nav_suff=["n", "g", "l"]),
    "063604": get_rinex_paths(rinex_dir, "65812800", nav_suff=["n", "g", "l"]),
}
files

{'1002': RinexPaths(obs_file=PosixPath('rinex/10022800.26o'), nav_files=(PosixPath('rinex/10022800.26n'), PosixPath('rinex/10022800.26g'), PosixPath('rinex/10022800.26l'))),
 '1005': RinexPaths(obs_file=PosixPath('rinex/10052800.26o'), nav_files=(PosixPath('rinex/10052800.26n'), PosixPath('rinex/10052800.26g'), PosixPath('rinex/10052800.26l'))),
 '1006': RinexPaths(obs_file=PosixPath('rinex/10062800.26o'), nav_files=(PosixPath('rinex/10062800.26n'), PosixPath('rinex/10062800.26g'), PosixPath('rinex/10062800.26l'))),
 '063604': RinexPaths(obs_file=PosixPath('rinex/65812800.26o'), nav_files=(PosixPath('rinex/65812800.26n'), PosixPath('rinex/65812800.26g'), PosixPath('rinex/65812800.26l')))}

## Sessioni - basi  (base - rover)

###  Sessione 1: 13:12 - 13:22 (LOCAL TIME)

* 1005 - 1002
* 1006 - 1002
* 063604 - 1005

###  Sessione 2: 13:23 - 13:33 (LOCAL TIME)

* 1005 - 1006
* 063604 - 1002
* 063604 - 1006

In [3]:
sessions = {
    "S1": (
        "2026/10/07 13:12:00",
        "2026/10/07 13:22:00",
        [("1005", "1002"), ("1006", "1002"), ("063604", "1005")],
    ),
    "S2": (
        "2026/10/07 13:23:00",
        "2026/10/07 13:33:00",
        [("1005", "1006"), ("063604", "1002"), ("063604", "1006")],
    ),
}

fuse_dt = timedelta(hours=2)  # local time (CEST) - UTC

In [4]:
# RTKLIB configuration
# (static, dual-frequency, short baselines). Everything else keeps RTKLIB's default.
conf = RtkConfig(
    # --- processing ---
    posmode="static",
    frequency="l1+l2",
    soltype="forward",  # last epoch = cumulative static solution
    elmask=15.0,  # deg
    ionoopt="brdc",  # double differences cancel most of it on short baselines
    tropopt="saas",
    sateph="precise",  # 0r "brdc" for broadcast ephemerides
    sat_pcv=True,  # satellite antenna PCO/PCV
    rcv_pcv=True,  # receiver antenna PCO/PCV (mixed antenna types)
    navsys="GRE",  # GPS + GLONASS + Galileo (1005 and 1006 log no Galileo)
    # --- ambiguity resolution ---
    armode="continuous",
    gloarmode="on",  # fix GLONASS ambiguities too: needed for 1005 in session 1
    arthres=3.0,  # ratio-test threshold
    arelmask=15.0,  # deg
    minfixsats=4,
    # --- antennas and base position ---
    rover_anttype="*",  # antenna type and height from the RINEX header
    base_anttype="*",
    base_postype="rinexhead",
    satantfile=PCV_file,
    rcvantfile=PCV_file,
    # --- output ---
    solformat="xyz",  # ECEF position + covariance
    outhead=True,
    timesys="gpst",
)

conf.write(Path("rtklib.conf"))

PosixPath('rtklib.conf')

## Manual test

In [5]:
# Define the start and end times for the processing and convert to GPST
datetime_start = "2026/10/07 13:12:00"
datetime_end = "2026/10/07 13:22:00"

time_start_gpst = utc_to_gpst(
    pd.Timestamp(datetime.strptime(datetime_start, "%Y/%m/%d %H:%M:%S") - fuse_dt)
)
time_end_gpst = utc_to_gpst(
    pd.Timestamp(datetime.strptime(datetime_end, "%Y/%m/%d %H:%M:%S") - fuse_dt)
)
path = run_rnx2rtkp(
    rover=files["1002"].obs_file,
    base=files["1005"].obs_file,
    nav=[files["1002"].nav_files, sp3_path, clk_path],
    config=conf,
    out=baseline_dir / "test_1002_1005.pos",
    start=time_start_gpst,
    end=time_end_gpst,
    tint=5,
)
pos = read_pos(path)

print("baseline 1002-1005")
print(f"start: {datetime.strftime(time_start_gpst, '%Y/%m/%d %H:%M:%S')} GPST")
print(f"end: {datetime.strftime(time_end_gpst, '%Y/%m/%d %H:%M:%S')} GPST")
print(
    f"baseline [dx, dy, dz]: {pos.iloc[-1]['x']:.3f}, {pos.iloc[-1]['y']:.3f}, {pos.iloc[-1]['z']:.3f} m"
)
print(pos.iloc[-1])

baseline 1002-1005
start: 2026/10/07 11:12:18 GPST
end: 2026/10/07 11:22:18 GPST
baseline [dx, dy, dz]: 4428543.651, 851920.528, 4495267.422 m
date              2026/10/07
time            11:22:15.000
x               4428543.6508
y                851920.5278
z               4495267.4215
Q                          1
ns                        11
sdx                   0.0026
sdy                   0.0006
sdz                   0.0016
sdxy                 -0.0006
sdyz                  0.0002
sdzx                  0.0018
age                      0.0
ratio                    9.2
t        2026-10-07 11:22:15
gps_s           1475407335.0
Name: 119, dtype: object


## Process all the baselines of each session (results saved in baselines/baselines.csv)

In [6]:
results = []
for session, (t0, t1, pairs) in sessions.items():
    start = utc_to_gpst(
        pd.Timestamp(datetime.strptime(t0, "%Y/%m/%d %H:%M:%S") - fuse_dt)
    )
    end = utc_to_gpst(
        pd.Timestamp(datetime.strptime(t1, "%Y/%m/%d %H:%M:%S") - fuse_dt)
    )
    for base, rover in pairs:
        r = process_baseline(
            rover,
            base,
            files[rover].obs_file,
            files[base].obs_file,
            [sp3_path, clk_path, files[rover].nav_files],
            conf,
            baseline_dir / f"{session}_{base}_{rover}.pos",
            start,
            end,
            tint=5,  # common 5s epochs: the receivers sample at 5s, 1s and 0.1s
            session=session,
        )
        results.append(r)


baselines = baselines_to_df([r for r in results if r])
baselines.to_csv(baseline_dir / "baselines.csv", index=False)
baselines

,session,base,rover,epoch,dx,dy,dz,cov_xx,cov_xy,cov_xz,cov_yy,cov_yz,cov_zz,length,Q,ns,ratio,pos_file
0,S1,1005,1002,2026-10-07 11:22:15,84.7704,-5100.2448,871.7748,0.000007,-3.600000e-07,3.240000e-06,3.600000e-07,4.000000e-08,2.560000e-06,5174.908148,1,11,9.2,baselines/S1_1005_1002.pos
1,S1,1006,1002,2026-10-07 11:22:15,-261.6419,-5224.5125,1234.8482,0.000002,4.000000e-08,1.210000e-06,2.500000e-07,1.600000e-07,1.000000e-06,5374.833711,1,11,4.2,baselines/S1_1006_1002.pos
2,S1,063604,1005,2026-10-07 11:22:15,-356.0602,-2145.9613,758.6197,0.000003,4.900000e-07,1.960000e-06,4.900000e-07,4.900000e-07,2.250000e-06,2303.786582,1,11,3.6,baselines/S1_063604_1005.pos
3,S2,1005,1006,2026-10-07 11:33:15,346.4366,124.2627,-363.0804,0.000001,3.600000e-07,1.000000e-06,3.600000e-07,3.600000e-07,1.210000e-06,516.997982,1,11,6.5,baselines/S2_1005_1006.pos
4,S2,063604,1002,2026-10-07 11:33:15,-271.2789,-7246.2125,1630.3871,0.000001,9.000000e-08,6.400000e-07,9.000000e-08,9.000000e-08,4.900000e-07,7432.317938,1,19,4.3,baselines/S2_063604_1002.pos
5,S2,063604,1006,2026-10-07 11:33:15,-9.6129,-2021.7015,395.5523,0.000002,6.400000e-07,1.960000e-06,4.900000e-07,6.400000e-07,1.960000e-06,2060.056064,1,12,6.8,baselines/S2_063604_1006.pos


In [7]:
from pyproj import Transformer

# Approximate rover position = base input position (the "ref pos" of the .pos file,
# i.e. the RINEX header position) + the estimated baseline vector.
base_xyz = np.vstack([read_ref_pos(f) for f in baselines["pos_file"]])
rover_xyz = base_xyz + baselines[["dx", "dy", "dz"]].to_numpy()

# ECEF (WGS84) -> geographic and UTM zone 32N
to_llh = Transformer.from_crs("EPSG:4978", "EPSG:4979", always_xy=True)
to_utm = Transformer.from_crs("EPSG:4978", "EPSG:32632", always_xy=True)
lon, lat, h = to_llh.transform(*rover_xyz.T)
east, north, _ = to_utm.transform(*rover_xyz.T)

approx_coords = pd.DataFrame({
    "point": baselines["rover"],
    "base": baselines["base"],
    "session": baselines["session"],
    "X": rover_xyz[:, 0],
    "Y": rover_xyz[:, 1],
    "Z": rover_xyz[:, 2],
    "lat": lat,
    "lon": lon,
    "h": h,
    "E": east,
    "N": north,
})
approx_coords


,point,base,session,X,Y,Z,lat,lon,h,E,N
0,1002,1005,S1,4.428544e+06,851920.5278,4.495267e+06,45.100330,10.888989,58.926836,648621.151986,4.995832e+06
1,1002,1006,S1,4.428544e+06,851920.5150,4.495267e+06,45.100329,10.888989,58.949738,648621.132550,4.995832e+06
2,1005,063604,S1,4.428461e+06,857023.4867,4.494397e+06,45.089159,10.952841,67.749416,653674.790330,4.994710e+06
3,1006,1005,S2,4.428805e+06,857145.0353,4.494033e+06,45.084540,10.953525,64.514615,653741.047428,4.994198e+06
4,1002,063604,S2,4.428546e+06,851923.2355,4.495269e+06,45.100324,10.889018,61.845874,648623.446554,4.995831e+06
5,1006,063604,S2,4.428807e+06,857147.7465,4.494034e+06,45.084535,10.953554,67.450404,653743.342828,4.994198e+06
